# Inference Optimization

Diffusion models are slow: 20-50 forward passes per image, each through a billion-param model. For video, multiply by frame count. This notebook benchmarks every major optimization and gives you concrete numbers. ~2-3 hrs, needs GPU.

In [ ]:
import torch
import time
from diffusers import StableDiffusionPipeline, DDIMScheduler, DPMSolverMultistepScheduler, EulerDiscreteScheduler
from torchmetrics.functional.multimodal import clip_score
from functools import partial
import matplotlib.pyplot as plt
import numpy as np
import gc

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")


def benchmark(pipe, prompt, num_steps=20, num_warmup=2, num_runs=5, **kwargs):
    """Benchmark a pipeline: returns avg latency, peak VRAM, generated image."""
    # Warmup
    for _ in range(num_warmup):
        _ = pipe(prompt, num_inference_steps=num_steps, **kwargs)

    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

    latencies = []
    for _ in range(num_runs):
        start = time.perf_counter()
        result = pipe(prompt, num_inference_steps=num_steps, **kwargs)
        torch.cuda.synchronize()
        latencies.append(time.perf_counter() - start)

    peak_vram = torch.cuda.max_memory_allocated() / 1e9
    avg_latency = np.mean(latencies)
    std_latency = np.std(latencies)
    print(f"  Latency: {avg_latency:.2f}s +/- {std_latency:.2f}s | Peak VRAM: {peak_vram:.2f} GB")
    return {"latency": avg_latency, "std": std_latency, "vram_gb": peak_vram, "image": result.images[0]}


prompt = "a professional photo of an astronaut riding a horse on the moon"
results = {}

In [ ]:
# EXERCISE: Implement a proper GPU benchmarking function

def benchmark(
    pipe,
    prompt: str,
    num_steps: int = 20,
    num_warmup: int = 2,
    num_runs: int = 5,
    **kwargs,
) -> dict:
    """Benchmark a diffusion pipeline with proper GPU synchronization.

    Must handle: warmup runs, CUDA synchronization, peak VRAM measurement.

    Returns:
        dict with: mean_latency_ms, std_latency_ms, peak_vram_gb, num_steps
    """
    # YOUR CODE
    raise NotImplementedError


In [ ]:
# Baseline: SD 1.5, FP32, 50 steps DDPM
# This is the worst-case scenario -- full precision, maximum steps, default scheduler.
# Expect ~5-10 GB VRAM, ~30-60s depending on GPU.

pipe_fp32 = StableDiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    torch_dtype=torch.float32,
    safety_checker=None,
).to(device)

print("Benchmarking FP32 / DDPM-50...")
results["FP32 / DDPM-50"] = benchmark(pipe_fp32, prompt, num_steps=50)

del pipe_fp32
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# FP16: half the memory, ~2x faster on modern GPUs (Tensor Cores).
# This is the single biggest easy win -- always do this first.

pipe_fp16 = StableDiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
    safety_checker=None,
).to(device)

print("Benchmarking FP16 / DDPM-50...")
results["FP16 / DDPM-50"] = benchmark(pipe_fp16, prompt, num_steps=50)

In [ ]:
# torch.compile: fuses ops, eliminates Python overhead, enables kernel-level optimizations.
# "reduce-overhead" mode is best for inference (uses CUDA graphs under the hood).
# First run after compile is slow (tracing), subsequent runs are faster.
# NOTE: Requires PyTorch 2.0+. May not work on all architectures.

pipe_fp16.unet = torch.compile(pipe_fp16.unet, mode="reduce-overhead")

print("Benchmarking FP16 / compiled / DDPM-50 (extra warmup for compilation)...")
results["FP16 / compiled / DDPM-50"] = benchmark(pipe_fp16, prompt, num_steps=50, num_warmup=3)

In [ ]:
# Scheduler comparisons (all with FP16 + compiled UNet)
# The scheduler determines how many denoising steps are needed.
# DDPM: original, needs 50-1000 steps.
# DDIM: deterministic, works well at 20-50 steps.
# DPM-Solver++: state-of-the-art ODE solver, good at 15-25 steps.
# Euler: simple, fast, good at 20-30 steps.

# DDIM 20 steps
pipe_fp16.scheduler = DDIMScheduler.from_config(pipe_fp16.scheduler.config)
print("Benchmarking FP16 / DDIM-20...")
results["FP16 / DDIM-20"] = benchmark(pipe_fp16, prompt, num_steps=20)

# DPM-Solver++ 20 steps
pipe_fp16.scheduler = DPMSolverMultistepScheduler.from_config(pipe_fp16.scheduler.config)
print("Benchmarking FP16 / DPM++-20...")
results["FP16 / DPM++-20"] = benchmark(pipe_fp16, prompt, num_steps=20)

# Euler 20 steps
pipe_fp16.scheduler = EulerDiscreteScheduler.from_config(pipe_fp16.scheduler.config)
print("Benchmarking FP16 / Euler-20...")
results["FP16 / Euler-20"] = benchmark(pipe_fp16, prompt, num_steps=20)

In [ ]:
# VAE tiling: splits the VAE decode into tiles to reduce peak memory.
# Essential for high-res generation where the full latent doesn't fit in VRAM.
# Trades a small amount of latency for significant memory savings.

pipe_fp16.enable_vae_tiling()

# Generate at higher resolution to show memory savings
print("Benchmarking FP16 / VAE-tiled / Euler-20 at 768x768...")
results["FP16 / VAE-tiled / Euler-20"] = benchmark(
    pipe_fp16, prompt, num_steps=20, height=768, width=768
)

pipe_fp16.disable_vae_tiling()

In [ ]:
# Combine best optimizations: FP16 + compiled + DPM-Solver++ 20 steps
# This should be the fastest configuration with acceptable quality.

pipe_fp16.scheduler = DPMSolverMultistepScheduler.from_config(pipe_fp16.scheduler.config)
# UNet is already compiled from earlier -- torch.compile is persistent
# Re-compile if needed (no-op if already compiled)
if not hasattr(pipe_fp16.unet, "_orig_mod"):
    pipe_fp16.unet = torch.compile(pipe_fp16.unet, mode="reduce-overhead")

print("Benchmarking FP16 / compiled / DPM++-20 (best combo)...")
results["FP16 / compiled / DPM++-20 (best)"] = benchmark(
    pipe_fp16, prompt, num_steps=20, num_warmup=3
)

In [ ]:
# CLIP score quality check: ensures optimizations haven't degraded output quality.
# CLIP score measures text-image alignment (higher = better).
# We want to see that faster configs don't sacrifice too much quality.

from torchmetrics.functional.multimodal import clip_score
from functools import partial

clip_score_fn = partial(clip_score, model_name_or_path="openai/clip-vit-base-patch16")

quality_scores = {}
for name, r in results.items():
    img_tensor = torch.from_numpy(np.array(r["image"])).permute(2, 0, 1).unsqueeze(0)
    score = clip_score_fn(img_tensor, [prompt]).item()
    quality_scores[name] = score
    print(f"{name}: CLIP score = {score:.2f}")

In [ ]:
import pandas as pd

# Build summary table
rows = []
for name, r in results.items():
    rows.append({
        "Config": name,
        "Latency (s)": f"{r['latency']:.2f}",
        "VRAM (GB)": f"{r['vram_gb']:.2f}",
        "CLIP Score": f"{quality_scores.get(name, 'N/A')}",
    })
df = pd.DataFrame(rows)
display(df)

# Bar charts
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
configs = [r["Config"] for r in rows]

# Latency
axes[0].barh(configs, [float(r["Latency (s)"]) for r in rows], color="steelblue")
axes[0].set_xlabel("Latency (seconds)")
axes[0].set_title("Generation Latency")

# VRAM
axes[1].barh(configs, [float(r["VRAM (GB)"]) for r in rows], color="coral")
axes[1].set_xlabel("Peak VRAM (GB)")
axes[1].set_title("Memory Usage")

# Quality
clip_vals = [float(r["CLIP Score"]) if r["CLIP Score"] != "N/A" else 0 for r in rows]
axes[2].barh(configs, clip_vals, color="seagreen")
axes[2].set_xlabel("CLIP Score")
axes[2].set_title("Quality (higher = better)")

plt.tight_layout()
plt.show()

# Speedup relative to baseline
baseline_latency = float(rows[0]["Latency (s)"])
print("\nSpeedup vs FP32/DDPM-50 baseline:")
for r in rows:
    speedup = baseline_latency / float(r["Latency (s)"])
    print(f"  {r['Config']}: {speedup:.1f}x")

## Video Inference

For video: multiply all bottlenecks by num_frames. A 16-frame video at 50 steps = 800 UNet forward passes. Sliding Tile Attention (STA) specifically addresses the attention bottleneck for video by using 3D tiled attention patterns instead of full attention, achieving comparable quality with much less memory. This is directly relevant to the long context problem (Notebook 12).

## Checkpoint

**Scenario:** new model is 3x slower than target. Optimization playbook:

1. **Profile first** -- is it compute-bound or memory-bound?
2. **Low-hanging fruit:** FP16, better scheduler (DDIM/DPM++), `torch.compile`.
3. **Architectural:** distillation (consistency models, LCM), attention optimization (STA, flash attention).
4. **Systems:** batching, model parallelism, quantization.

Always measure quality alongside speed -- there's no free lunch.

**Further reading:** LCM ([2310.04378](https://arxiv.org/abs/2310.04378)), STA ([2502.04507](https://arxiv.org/abs/2502.04507)).